# bench-imai-artifact — ArtiFact Multi-Generator Detection Benchmark

This notebook follows up on the CIFAKE benchmark (`bench-imai`) by training on the
**ArtiFact** dataset (200×200, 25 generators) and evaluating cross-dataset
generalisation on CIFAKE and the Shoes dataset. We use a focused subset of the
`timm` backbones from `pipelines_torch` registries — chosen based on the findings
from `bench-imai` (best in-domain, best generaliser, and recommended next steps).

GPU is required for the heavier backbones.

Note: See the final section **"9. Analysis and Lessons Learned"** for a
concise summary of results, cross-dataset generalisation findings and recommended
next steps.

## 1. Environment setup
Install optional vision dependencies (`timm`, `kaggle`) so the registry backbones
load without manual package management.

In [ ]:
%rm -rf bench_research_ml_project
!git clone https://github.com/oremaz/bench_research_ml_project
%cd bench_research_ml_project
!pip install -r requirements.txt
!pip uninstall numpy scipy scikit-learn -y
!pip install numpy==1.24.3 scipy==1.10.1 scikit-learn==1.3.0
!pip install -q timm==1.0.9 kaggle rich

In [ ]:
%cd ml_pipeline

## 2. Imports and deterministic utilities
Everything important (models, metrics, benchmarking) is imported from the shared
library so we avoid redefining models or augmentations inside the notebook.

In [ ]:
import os
from pathlib import Path
from typing import Iterable
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.vision_models import MODEL_REGISTRY as VISION_MODELS
from pipelines_torch.base import SimplePredictor
from utils.metrics import METRIC_REGISTRY
from utils.utils import load_model_by_name, RESULTS_DIR
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Download and prepare ArtiFact
The ArtiFact dataset contains ~2.5M images (965K real + 1.5M fake) from 25 different
generators at 200×200 resolution. This provides much more generator diversity than CIFAKE
(single generator). We use stratified 15% sampling to keep training tractable.

In [ ]:
from pathlib import Path

from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_ARTIFACT = "awsaf49/artifact-dataset"
DATA_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_ARTIFACT,
    local_dir=Path("data/artifact-real-fake"),
    description="ArtiFact dataset",
    kaggle_subdir="artifact-dataset",
)

if DATA_DIR.exists():
    print(f"✅ ArtiFact data available at {DATA_DIR}")
else:
    print(f"⚠️ ArtiFact dataset missing at {DATA_DIR}")

In [ ]:
IMG_SIZE = 200
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def is_imagefolder_dir(path: Path) -> bool:
    path = Path(path)
    if not path.is_dir():
        return False
    subdirs = [p for p in path.iterdir() if p.is_dir()]
    if len(subdirs) < 2:
        return False
    return any(any(child.is_file() for child in d.iterdir()) for d in subdirs)

def find_imagefolder_split(base_dir: Path) -> Path:
    base_dir = Path(base_dir)
    if is_imagefolder_dir(base_dir):
        return base_dir
    preferred = ("train", "test", "validation", "val", "eval")
    for name in preferred:
        for variant in {name, name.upper(), name.capitalize()}:
            candidate = base_dir / variant
            if is_imagefolder_dir(candidate):
                return candidate
    for candidate in sorted(base_dir.rglob("*")):
        if is_imagefolder_dir(candidate):
            return candidate
    raise ValueError(f"Could not locate an ImageFolder split inside {base_dir}")

train_split = find_imagefolder_split(DATA_DIR)
print(f"Using ArtiFact data from: {train_split}")

train_ds = datasets.ImageFolder(train_split, transform=transform)

class_names = train_ds.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")
print(f"Total samples: {len(train_ds)}")

In [ ]:
def dataset_to_numpy(dataset: datasets.ImageFolder):
    tensors = [img for img, _ in dataset]
    X = torch.stack(tensors).numpy()
    y = np.array(dataset.targets, dtype=np.int64)
    return X.astype(np.float32), y

X_all, y_all = dataset_to_numpy(train_ds)

SAMPLE_FRACTION = 0.15
print(f"Original ArtiFact size: {len(X_all)} images")

X_all, _, y_all, _ = train_test_split(
    X_all, y_all,
    train_size=SAMPLE_FRACTION,
    random_state=SEED,
    stratify=y_all,
)
print(f"Training data shape: {X_all.shape}")
print(f"Class distribution: {np.bincount(y_all)}")

## 4. Configure `BenchmarkRunner`
We select a focused subset of models based on the `bench-imai` findings:
- **binary_efficientnet_b4_ns** — best in-domain accuracy on CIFAKE (97.1%)
- **clip_classifier** — best cross-dataset generaliser (79.1% on Shoes)
- **timm_convnextv2_tiny** — strong modern ConvNet backbone
- **timm_dinov2_vit_base** — DINOv2 self-supervised ViT (recommended next step)
- **resnet50** — standard baseline for comparison

In [ ]:
SELECTED_MODELS = [
    "binary_efficientnet_b4_ns",
    "clip_classifier",
    "timm_convnextv2_tiny",
    "timm_dinov2_vit_base",
    "resnet50",
]

model_configs = []
epochs = {}
for name in SELECTED_MODELS:
    if name not in VISION_MODELS:
        raise KeyError(f"{name} is not registered in pipelines_torch.vision_models")
    model_configs.append({
        "name": name,
        "class": VISION_MODELS[name],
        "params": {"num_classes": num_classes},
    })
    epochs[name] = 7


runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    task_type="classification",
    device=DEVICE,
    epochs=epochs,
    batch_size=32,
    early_stopping=None,
    use_class_weights=True,
    use_kfold=False,
    learning_rate=3e-4,
    path_start="bench_imai_artifact",
    random_state=SEED,
)
results_df = runner.run(X_all, y_all)
results_df

In [ ]:
import os
import tarfile
from IPython.display import FileLink

os.chdir('/kaggle/working/results')

with tarfile.open('/kaggle/working/all_files.tar.gz', 'w:gz') as tar:
    for root, dirs, files in os.walk('.'):
        for file in files:
            tar.add(os.path.join(root, file))

os.chdir('/kaggle/working')
FileLink('all_files.tar.gz')

## 6. Evaluate the ArtiFact test split
Re-use the same helper to score a held-out ArtiFact split.

In [ ]:
def evaluate_saved_models(model_names: Iterable[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
    }
    records = []
    for name in model_names:
        try:
            model = load_model_by_name(VISION_MODELS[name], name, {"num_classes": num_classes}, path_start="bench-imai-artifact-final/bench_imai_artifact")
        except FileNotFoundError:
            print(f"⚠️ Skipping {name}: checkpoint not found")
            continue
        print(name)
        predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=32)
        probs = predictor.predict_proba(X)
        scores = {
            label: float(METRIC_REGISTRY[key](y, probs))
            for label, key in metric_map.items()
        }
        records.append({
            "model": name,
            **scores,
        })
    return pd.DataFrame.from_records(records)

In [ ]:
test_candidates = [DATA_DIR / "test", DATA_DIR / "Test", DATA_DIR / "val", DATA_DIR / "validation"]
artifact_test_dir = None
for candidate in test_candidates:
    if candidate.exists() and any(candidate.iterdir()):
        try:
            test_ds = datasets.ImageFolder(candidate, transform=transform)
            if len(test_ds) > 0:
                artifact_test_dir = candidate
                break
        except Exception:
            continue

if artifact_test_dir is not None:
    print(f"Found ArtiFact test split at: {artifact_test_dir}")
    test_ds = datasets.ImageFolder(artifact_test_dir, transform=transform)
    X_test_artifact, y_test_artifact = dataset_to_numpy(test_ds)
else:
    print("No test split found, creating 20% hold-out from training data")
    indices = np.arange(len(X_all))
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=SEED, stratify=y_all
    )
    X_test_artifact = X_all[test_idx]
    y_test_artifact = y_all[test_idx]
    X_all = X_all[train_idx]
    y_all = y_all[train_idx]

print(f"ArtiFact test shape: {X_test_artifact.shape}")

In [ ]:
artifact_test_metrics = evaluate_saved_models(SELECTED_MODELS, X_test_artifact, y_test_artifact)
artifact_test_metrics.sort_values("accuracy", ascending=False)

## 7. CIFAKE cross-dataset generalisation check
Test whether ArtiFact-trained models transfer to the lower-resolution CIFAKE
test set (32×32, single generator).

In [ ]:
from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_CIFAKE = "birdy654/cifake-real-and-ai-generated-synthetic-images"
CIFAKE_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_CIFAKE,
    local_dir=Path("data/cifake"),
    description="CIFAKE dataset",
    kaggle_subdir="cifake-real-and-ai-generated-synthetic-images",
)

CIFAKE_SIZE = 32
cifake_transform = transforms.Compose([
    transforms.Resize((CIFAKE_SIZE, CIFAKE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cifake_test_dir = CIFAKE_DIR / "test"
if not cifake_test_dir.exists():
    raise FileNotFoundError("Expected CIFAKE to expose a test/ split")

cifake_test_ds = datasets.ImageFolder(cifake_test_dir, transform=cifake_transform)
X_test_cifake, y_test_cifake = dataset_to_numpy(cifake_test_ds)
print(f"CIFAKE test shape: {X_test_cifake.shape}")

cifake_test_metrics = evaluate_saved_models(SELECTED_MODELS, X_test_cifake, y_test_cifake)
cifake_test_metrics.sort_values("accuracy", ascending=False)

## 8. Shoes dataset generalisation check
The Shoes dataset tests a different distribution shift: high-resolution images
from a single generator (Midjourney).

In [ ]:
from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_SHOES = "sunnykakar/shoes-dataset-real-and-ai-generated-images"
SHOES_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_SHOES,
    local_dir=Path("data/shoes-ai-vs-real"),
    description="SunnyKakar Shoes (AI vs Real)",
    kaggle_subdir="shoes-dataset-real-and-ai-generated-images",
)

shoes_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

shoes_split = find_imagefolder_split(SHOES_DIR)
print(f"Using {shoes_split} for evaluation")

shoes_ds = datasets.ImageFolder(shoes_split, transform=shoes_transform)
X_test_shoes, y_test_shoes = dataset_to_numpy(shoes_ds)
print(f"Shoes test shape: {X_test_shoes.shape}")

shoes_metrics = evaluate_saved_models(SELECTED_MODELS, X_test_shoes, y_test_shoes)
shoes_metrics.sort_values("accuracy", ascending=False)

# 9. Analysis and Lessons Learned

## Overview of Results

This benchmark evaluated **5 vision models** on AI-generated vs. Real image detection,
trained on the ArtiFact dataset (200×200, 25 generators) and tested across three datasets:
- **ArtiFact (In-Domain)**: Multi-generator, 200×200 images
- **CIFAKE (Cross-Dataset)**: 32×32, single generator (StyleGAN)
- **Shoes (Cross-Dataset)**: High-resolution, Midjourney-generated

### Model Selection Rationale

The five models were chosen based on the `bench-imai` findings:

| Model | Selection Reason |
|-------|------------------|
| **binary_efficientnet_b4_ns** | Best in-domain accuracy on CIFAKE (97.1%) |
| **clip_classifier** | Best cross-dataset generaliser (79.1% on Shoes) |
| **timm_convnextv2_tiny** | Strong modern ConvNet (96.0% on CIFAKE) |
| **timm_dinov2_vit_base** | DINOv2 self-supervised ViT — recommended next step |
| **resnet50** | Standard baseline for comparison |

---

## Key Questions This Benchmark Answers

1. **Does multi-generator training improve generalisation?** — Compare ArtiFact→Shoes
   drop vs. the 15–40% drops observed when training on CIFAKE alone.
2. **Does DINOv2 live up to its promise?** — Self-supervised pretraining on diverse
   data should provide more robust representations.
3. **Resolution mismatch impact** — ArtiFact trains at 200×200; CIFAKE tests at 32×32.
   How much does downscaling hurt?

---

## Expected Improvements over bench-imai

- **Reduced single-generator overfitting**: ArtiFact’s 25 generators should prevent
  models from memorising StyleGAN-specific artefacts.
- **Better Shoes transfer**: Models should see smaller accuracy drops on the Shoes
  dataset compared to the 15–40% drops from CIFAKE-trained models.
- **DINOv2 baseline**: First evaluation of DINOv2 on this detection task.

---

## Next Steps

1. **Ensemble CLIP + EfficientNet** — Combine the best generaliser with the best
   in-domain model.
2. **Resolution-adaptive training** — Progressive fine-tuning from 32×32 → 200×200
   → 512×512 to build resolution-invariant features.
3. **Frequency-domain features** — FFT-based fake detection to complement spatial
   CNN features.
4. **Expand to GenImage / FaceForensics++** — Broader benchmark coverage across
   domains (faces, scenes, text-to-image).